In [5]:
from pathlib import Path

# Import pyarrow before pandas so pandas does not hit duplicate Arrow extension
# registration when the kernel has been partially initialized.
import pyarrow.parquet as pq
import pandas as pd

pipeline_name = "strongestPath"

# Support running notebook from either repo root or ml/polys/workbook.
candidate_parquet_roots = [
    Path("ml/polys/artifacts/parquet"),
    Path("../artifacts/parquet"),
    Path("artifacts/parquet"),
]

parquet_root = next((p for p in candidate_parquet_roots if p.exists()), None)
if parquet_root is None:
    checked = "\n".join(str(p.resolve()) for p in candidate_parquet_roots)
    raise FileNotFoundError(
        "Could not locate parquet artifacts root. Checked:\n"
        f"{checked}\n"
        "Run spark-submit first so artifacts are generated."
    )

base = parquet_root / pipeline_name
latest_file = base / "latest.txt"

if not latest_file.exists():
    raise FileNotFoundError(
        f"Could not find latest run pointer at {latest_file.resolve()}. "
        "Run spark-submit first so artifacts are generated."
    )

run_id = latest_file.read_text(encoding="utf-8").strip()
if not run_id:
    raise ValueError(f"latest.txt exists but is empty: {latest_file.resolve()}")

run_base = base / run_id
print(f"pipeline={pipeline_name}")
print(f"parquet_root={parquet_root.resolve()}")
print(f"run_id={run_id}")
print(f"run_base={run_base.resolve()}")

pipeline=strongestPath
parquet_root=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet
run_id=20260421_163935
run_base=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935


In [6]:
stages = [
    "extractS12",
    "groupedDegreeSum",
    "rowScalarDegreePivot",
    "aggregatedScalarDegreePivot",
    "reducedResult",
]


def read_parquet_dir(stage_path: Path) -> pd.DataFrame:
    """Read Spark-written Parquet without pandas' pyarrow engine (avoids extension registration bugs)."""
    try:
        table = pq.read_table(str(stage_path))
    except ImportError as err:
        raise ImportError(
            "pyarrow is required to read these Parquet files. "
            "Install with: pip install pyarrow"
        ) from err
    return table.to_pandas()


def load_stage(stage_name: str) -> pd.DataFrame:
    stage_path = run_base / stage_name
    if not stage_path.exists():
        raise FileNotFoundError(f"Missing stage output: {stage_path.resolve()}")
    return read_parquet_dir(stage_path)


datasets = {stage: load_stage(stage) for stage in stages}
print(f"Loaded {len(datasets)} stage datasets for run {run_id} (via pyarrow.parquet).")

Loaded 5 stage datasets for run 20260421_163935 (via pyarrow.parquet).


In [7]:
for stage_name, frame in datasets.items():
    stage_path = run_base / stage_name
    print(f"\n[{stage_name}] path={stage_path.resolve()}")
    print(f"rows={len(frame)}, columns={list(frame.columns)}")
    display(frame.head())


[extractS12] path=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935\extractS12
rows=114, columns=['index', 'maxN', 'rowScalar', 'divisor', 'scalar', 'degree']


,index,maxN,rowScalar,divisor,scalar,degree
0,2,8,1,1,3,-1
1,2,8,1,1,42,0
2,2,8,1,1,56,0
3,2,8,1,1,-6,1
4,2,8,1,1,-7,1



[groupedDegreeSum] path=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935\groupedDegreeSum
rows=24, columns=['maxN', 'index', 'degree', 'scalar_result']


,maxN,index,degree,scalar_result
0,8,2,-1,3.0000000000
1,8,2,0,98.0000000000
2,8,2,1,-28.0000000000
3,8,2,2,2.0000000000
4,8,3,-1,4.0000000000



[rowScalarDegreePivot] path=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935\rowScalarDegreePivot
rows=21, columns=['N', 'rowScalar_num', '-1', '0', '1', '2']


,N,rowScalar_num,-1,0,1,2
0,2,1,3.0000000000,98.0000000000,-28.0000000000,2.0000000000
1,3,1,4.0000000000,72.0000000000,-24.0000000000,2.0000000000
2,3,2,None,56.0000000000,-15.0000000000,1.0000000000
3,4,1,5.0000000000,50.0000000000,-20.0000000000,2.0000000000
4,4,2,None,42.0000000000,-13.0000000000,1.0000000000



[aggregatedScalarDegreePivot] path=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935\aggregatedScalarDegreePivot
rows=6, columns=['N', '-1', '0', '1', '2']


,N,-1,0,1,2
0,2,3.0000000000,98.0000000000,-28.0000000000,2.0000000000
1,3,4.0000000000,184.0000000000,-54.0000000000,4.0000000000
2,4,5.0000000000,358.0000000000,-106.0000000000,8.0000000000
3,5,6.0000000000,708.0000000000,-210.0000000000,16.0000000000
4,6,7.0000000000,1410.0000000000,-418.0000000000,32.0000000000



[reducedResult] path=C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\artifacts\parquet\strongestPath\20260421_163935\reducedResult
rows=6, columns=['maxN', 'index', 'index_result']


,maxN,index,index_result
0,8,2,2.375
1,8,3,8.500
2,8,4,22.625
3,8,5,52.750
4,8,6,114.875


In [8]:
reduced_result = datasets["reducedResult"]

if "index_result" in reduced_result.columns:
    display(reduced_result.sort_values("index_result", ascending=False).head(20))
else:
    print("Column 'index_result' was not found in reducedResult.")

,maxN,index,index_result
5,8,7,241.000
4,8,6,114.875
3,8,5,52.750
2,8,4,22.625
1,8,3,8.500
0,8,2,2.375


In [ ]:
# If you ever see Arrow extension registration errors, restart the kernel once
# (they can accumulate after repeated imports in one long-lived session).

## Cross-check against legacy `s12QuadQ` Cypher

Before the polys Spark pipeline was written, quadratics were reconstructed directly from Neo4j via the Cypher in `ZADScriptsK/ZADScripts/src/zadscripts/DFScripts.java::s12QuadQ` (comment: *"Querry for returning a single quadratic"*). Running the equivalent Cypher against the same Neo4j graph is a sanity check that the Spark refactor preserves semantics.

**Math difference you must remember to interpret the comparison.**  
`s12QuadQ` folds the `degree = -1` remainder term into `degree = 0`:

- `TTScalar = vScalar * 2` when `vDegree == "-1"`, else `vScalar`
- `TTDegree = 0` when `vDegree == "-1"`, else `vDegree`

The polys Spark pipeline currently keeps `degree = -1` as a separate bucket. So:

- The `s12QuadQ`-equivalent output will **not contain any `degree = -1` rows**.
- For each `index`, its `degree = 0` value equals the Spark pipeline's `degree = 0` contribution *plus* `2 * (Spark pipeline's contribution at degree = -1)`.

Below we run the query via the `neo4j` Python driver (same driver pattern used by `ml/spark_graph_builder`), apply the same fold to the Spark extract for side-by-side comparison, and display the diff. Cells silently skip if `neo4j` is not installed.

In [9]:
try:
    from neo4j import GraphDatabase
    NEO4J_AVAILABLE = True
    print("neo4j Python driver available.")
except ImportError:
    NEO4J_AVAILABLE = False
    print("neo4j Python driver not installed. Cypher cross-check cells will be skipped.")
    print("To enable: pip install neo4j")

neo4j Python driver available.


In [10]:
s12quadq_df = None

if NEO4J_AVAILABLE:
    candidate_db_props = [
        Path("ml/polys/db.properties"),
        Path("../db.properties"),
        Path("db.properties"),
    ]
    db_props_file = next((p for p in candidate_db_props if p.exists()), None)

    if db_props_file is None:
        checked = "\n".join(str(p.resolve()) for p in candidate_db_props)
        raise FileNotFoundError(
            "Could not locate ml/polys/db.properties. Checked:\n" + checked
        )

    db_props = {}
    for line in db_props_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            db_props[k.strip()] = v.strip()

    # JDBC form in db.properties is e.g. "jdbc:neo4j:bolt://localhost".
    # Python driver expects plain Bolt URL with an explicit port.
    raw_url = db_props["neo4j.url"]
    bolt_url = raw_url.replace("jdbc:neo4j:", "")
    host_part = bolt_url.split("//", 1)[-1]
    if ":" not in host_part:
        bolt_url = f"{bolt_url}:7687"

    # Equivalent of ZADScripts/DFScripts.java::s12QuadQ without APOC.
    # TTScalar = vScalar*2 when vDegree=-1 else vScalar; TTDegree = 0 when vDegree=-1 else vDegree.
    cypher = """
    UNWIND range(toInteger($rangeLow), toInteger($rangeHigh)) AS n
    WITH toString(n) AS N, $nMax AS nMax
    MATCH (v:VertexNode)<-[:VertexIndexedBy]-(i:IndexedBy {N: N, MaxN: nMax, Dimension: "2"})-[:TwoFactor]->(t:TwoSeqFactor)
    RETURN
      i.N      AS iN,
      i.MaxN   AS iM,
      t.twoSeq AS rowScalar,
      "2"      AS tDivisor,
      toString(CASE WHEN toString(v.Degree) = "-1" THEN toInteger(v.Scalar) * 2 ELSE toInteger(v.Scalar) END) AS TTScalar,
      toString(CASE WHEN toString(v.Degree) = "-1" THEN 0                       ELSE toInteger(v.Degree) END) AS TTDegree
    ORDER BY iN, rowScalar, TTDegree
    """

    params = {"rangeLow": "2", "rangeHigh": "7", "nMax": "8"}

    driver = GraphDatabase.driver(
        bolt_url,
        auth=(db_props["neo4j.user"], db_props["neo4j.password"]),
    )
    try:
        with driver.session(database=db_props["neo4j.database"]) as session:
            result = session.run(cypher, params)
            s12quadq_df = pd.DataFrame([dict(r) for r in result])
    finally:
        driver.close()

    print(f"db_properties : {db_props_file.resolve()}")
    print(f"bolt_url      : {bolt_url}")
    print(f"database      : {db_props['neo4j.database']}")
    print(f"s12QuadQ rows : {len(s12quadq_df)}")
    display(s12quadq_df.head(10))
else:
    print("Skipped: neo4j driver not available.")

db_properties : C:\Users\tomas\JavaProjects\Aibeceles\ml\polys\db.properties
bolt_url      : bolt://localhost:7687
database      : twopolynomial
s12QuadQ rows : 114


,iN,iM,rowScalar,tDivisor,TTScalar,TTDegree
0,2,8,1,2,6,0
1,2,8,1,2,42,0
2,2,8,1,2,56,0
3,2,8,1,2,-6,1
4,2,8,1,2,-7,1
5,2,8,1,2,-7,1
6,2,8,1,2,-8,1
7,2,8,1,2,1,2
8,2,8,1,2,1,2
9,3,8,1,2,8,0


In [11]:
if NEO4J_AVAILABLE and s12quadq_df is not None and len(s12quadq_df) > 0:
    from decimal import Decimal

    # Apply the s12QuadQ fold to the Spark extract so it lines up row-for-row:
    #   if degree == -1 -> degree = 0, scalar = scalar * 2.
    spark_ext = datasets["extractS12"].copy()
    spark_ext["degree_folded"] = spark_ext["degree"].where(spark_ext["degree"] != "-1", "0")
    spark_ext["scalar_folded"] = [
        str(Decimal(s) * 2) if d == "-1" else s
        for s, d in zip(spark_ext["scalar"], spark_ext["degree"])
    ]

    # Sum (scalar_folded * rowScalar) per (maxN, index, degree_folded):
    spark_ext["contrib"] = [
        Decimal(s) * Decimal(r)
        for s, r in zip(spark_ext["scalar_folded"], spark_ext["rowScalar"])
    ]
    spark_agg = (
        spark_ext.groupby(["maxN", "index", "degree_folded"], as_index=False)["contrib"]
        .sum()
        .rename(columns={"degree_folded": "degree", "contrib": "spark_result"})
    )

    # Same aggregation from the s12QuadQ result (sum TTScalar * rowScalar per (iM, iN, TTDegree)):
    quadq = s12quadq_df.copy()
    quadq["contrib"] = [
        Decimal(s) * Decimal(r)
        for s, r in zip(quadq["TTScalar"], quadq["rowScalar"])
    ]
    quadq_agg = (
        quadq.groupby(["iM", "iN", "TTDegree"], as_index=False)["contrib"]
        .sum()
        .rename(
            columns={
                "iM": "maxN",
                "iN": "index",
                "TTDegree": "degree",
                "contrib": "s12quadq_result",
            }
        )
    )

    compare = spark_agg.merge(quadq_agg, on=["maxN", "index", "degree"], how="outer")
    compare["match"] = compare["spark_result"].astype(str) == compare["s12quadq_result"].astype(str)
    compare["degree_int"] = compare["degree"].astype(int)
    compare = compare.sort_values(["maxN", "index", "degree_int"]).drop(columns=["degree_int"])

    all_match = bool(compare["match"].all())
    mismatch_count = int((~compare["match"]).sum())

    print(f"total rows compared : {len(compare)}")
    print(f"all rows match      : {all_match}")
    print(f"mismatch rows       : {mismatch_count}")
    display(compare)
else:
    print("Skipped: no s12QuadQ data to compare.")

total rows compared : 18
all rows match      : True
mismatch rows       : 0


,maxN,index,degree,spark_result,s12quadq_result,match
0,8,2,0,104,104,True
1,8,2,1,-28,-28,True
2,8,2,2,2,2,True
3,8,3,0,192,192,True
4,8,3,1,-54,-54,True
5,8,3,2,4,4,True
6,8,4,0,368,368,True
7,8,4,1,-106,-106,True
8,8,4,2,8,8,True
9,8,5,0,720,720,True


### Interpreting the comparison

- **All rows match** (`all_match == True`): the polys Spark pipeline reconstructs the same quadratics the legacy `s12QuadQ` Cypher produced, once the `degree=-1` fold is applied. The refactor is semantically equivalent at this stage.
- **Mismatches present**: first check the unmatched rows in `compare`. Common causes:
    1. Missing rows on one side → Cypher used different `rangeLow/rangeHigh/nMax` than the Spark run. Adjust the `params` dict in the previous cell to match the spark-submit args.
    2. Scalar difference → likely a math-semantics drift; compare against the raw `extractS12` rows and the `s12QuadQ` source in `DFScripts.java` to find the discrepant step.
    3. `degree` mismatch → the fold rule changed; re-check the `CASE WHEN toString(v.Degree) = "-1"` branches in the Cypher.

This cell reports `spark_result` vs `s12quadq_result` for every `(maxN, index, degree)` key in either dataset, so any asymmetry is visible at a glance.

### `Q_folded` as canonical: per-row contributions

Going forward, treat **`Q_folded`** (the `s12QuadQ` fold: `degree = -1` becomes `0` with `scalar *= 2`) as the quadratic you care about, not the raw Laurent `Q_raw` with a separate `x^{-1}` column.

In **`extractS12`**, each **row** is one summand of the expanded form: weighted coefficient `(scalar * rowScalar / divisor)` at Laurent degree `degree`. After the fold, that row contributes

`(scalar_folded * rowScalar / divisor) * maxN**degree_folded`

to **`Q_folded(maxN)`** for its pipeline `index` (same integer as pivot column **`N`**).

**Sum over all rows** sharing the same `(maxN, index)` equals the finished **`Q_folded(maxN)`** for that quadratic. For the `maxN = 8` strongest-path artifacts used in this workbook, that value matches **`2^(N+1)`** for each `N` in the run (see the cell above).

The next cell lists every row’s `row_eval` and checks the per-`index` sum against both the aggregated-pivot `Q_folded` and `2^(N+1)`.

In [ ]:
from decimal import Decimal
import math

ext = datasets["extractS12"].copy()
ext["index"] = ext["index"].astype(int)
ext["maxN_int"] = ext["maxN"].astype(int)

deg_folded: list[int] = []
sc_folded: list[Decimal] = []
for d, s in zip(ext["degree"], ext["scalar"]):
    if str(d) == "-1":
        deg_folded.append(0)
        sc_folded.append(Decimal(str(s)) * 2)
    else:
        deg_folded.append(int(d))
        sc_folded.append(Decimal(str(s)))

ext["degree_folded"] = deg_folded
ext["scalar_folded"] = sc_folded

row_evals: list[Decimal] = []
for i in range(len(ext)):
    rs = Decimal(str(ext["rowScalar"].iloc[i]))
    dv = Decimal(str(ext["divisor"].iloc[i]))
    if dv == 0:
        raise ValueError("extractS12 row has divisor 0")
    m = int(ext["maxN_int"].iloc[i])
    row_evals.append((rs / dv) * sc_folded[i] * (Decimal(m) ** deg_folded[i]))

ext["row_eval"] = row_evals
ext["row_ord"] = ext.groupby(["maxN_int", "index"], sort=False).cumcount() + 1

show_cols = [
    "maxN_int",
    "index",
    "row_ord",
    "rowScalar",
    "divisor",
    "scalar",
    "degree",
    "degree_folded",
    "scalar_folded",
    "row_eval",
]
print("Each extract row → folded summand evaluated at x = maxN (column row_eval)")
display(ext[show_cols].sort_values(["maxN_int", "index", "row_ord"]).reset_index(drop=True))

per_key = (
    ext.groupby(["maxN_int", "index"], as_index=False)
    .agg(sum_row_eval=("row_eval", lambda xs: sum(xs, start=Decimal(0))), n_rows=("row_eval", "count"))
)
per_key["sum_row_eval_f"] = per_key["sum_row_eval"].map(float)

agg = datasets["aggregatedScalarDegreePivot"].copy()
agg["N"] = agg["N"].astype(int)


def q_folded_at_maxn(row, max_n: int) -> float:
    a2 = float(row.get("2") or 0)
    a1 = float(row.get("1") or 0)
    a0 = float(row.get("0") or 0)
    am1 = float(row.get("-1") or 0)
    return a2 * max_n**2 + a1 * max_n + (a0 + 2 * am1)


ref_rows = []
for _, r in agg.sort_values("N").iterrows():
    n = int(r["N"])
    max_n = int(per_key.loc[per_key["index"] == n, "maxN_int"].iloc[0])
    ref_rows.append({"index": n, "q_folded_agg": q_folded_at_maxn(r, max_n)})

ref = pd.DataFrame(ref_rows)
summary = per_key.merge(ref, on="index", how="left")
summary["expected_pow2"] = summary["index"].map(lambda n: float(2 ** (n + 1)))
summary["match_agg"] = summary.apply(
    lambda r: math.isclose(r["sum_row_eval_f"], r["q_folded_agg"], rel_tol=0.0, abs_tol=1e-6),
    axis=1,
)
summary["match_pow2"] = summary.apply(
    lambda r: math.isclose(r["sum_row_eval_f"], r["expected_pow2"], rel_tol=0.0, abs_tol=1e-6),
    axis=1,
)

print("Per (maxN, index): sum of row_eval vs aggregatedScalarDegreePivot Q_folded vs 2^(N+1)")
display(
    summary[
        [
            "maxN_int",
            "index",
            "n_rows",
            "sum_row_eval_f",
            "q_folded_agg",
            "expected_pow2",
            "match_agg",
            "match_pow2",
        ]
    ]
)

if not bool(summary["match_agg"].all()):
    raise AssertionError("Row-sum Q_folded does not match aggregatedScalarDegreePivot; inspect extractS12 / fold.")
if not bool(summary["match_pow2"].all()):
    print("Note: sum_row_eval != 2^(N+1) for some index — expected if maxN or pipeline differs from this notebook’s check.")
else:
    print("All indices: sum(row_eval) == Q_folded(agg) == 2^(N+1) for this run.")

In [13]:
import math

agg = datasets["aggregatedScalarDegreePivot"].copy()
agg["N"] = agg["N"].astype(int)
for _, row in agg.sort_values("N").iterrows():
    N  = int(row["N"])
    a2 = float(row.get("2") or 0)
    a1 = float(row.get("1") or 0)
    a0 = float(row.get("0") or 0)
    am1 = float(row.get("-1") or 0)
    maxN = 8
    Q_raw    = a2*maxN**2 + a1*maxN + a0 + am1/maxN
    Q_folded = a2*maxN**2 + a1*maxN + (a0 + 2*am1)
    is_pow2 = (Q_folded > 0) and abs(Q_folded - 2**round(math.log2(Q_folded))) < 1e-9
    print(f"N={N}: Q_raw(maxN)={Q_raw}  Q_folded(maxN)={Q_folded}  power_of_2={is_pow2}")

N=2: Q_raw(maxN)=2.375  Q_folded(maxN)=8.0  power_of_2=True
N=3: Q_raw(maxN)=8.5  Q_folded(maxN)=16.0  power_of_2=True
N=4: Q_raw(maxN)=22.625  Q_folded(maxN)=32.0  power_of_2=True
N=5: Q_raw(maxN)=52.75  Q_folded(maxN)=64.0  power_of_2=True
N=6: Q_raw(maxN)=114.875  Q_folded(maxN)=128.0  power_of_2=True
N=7: Q_raw(maxN)=241.0  Q_folded(maxN)=256.0  power_of_2=True
